# Autoencoder postprocessing

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/AUTOENCODER")

## Training history

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.set(ylabel='Reconstruction loss', xlabel='epoch', yscale='log')

df = pd.read_csv(output_dir / "history.csv")
df.plot(y='train_loss', label='Training', ax=ax)
df.plot(y='validation_loss', label='Validation', ax=ax)

## Model loading

In [ ]:
import torch
from torch.utils.data import DataLoader

from modules.model import Autoencoder
from modules.dataset import LogMinMaxScale, EnsembleDataset

checkpoint = torch.load(output_dir / 'model.pt', map_location='cpu')

## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
scale     = checkpoint['SCALE']
transform = LogMinMaxScale(min_value, max_value, scale)

# Model
model = Autoencoder(checkpoint['LATENT_DIM'], in_shape=checkpoint['IN_SHAPE'])
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
#
# Model summary
#
from torchsummary import summary
H,W = checkpoint['IN_SHAPE']
summary(model, (1,H,W))

In [ ]:
device = torch.device('cpu')

## Reconstruction of the validation dataset

In [ ]:
#
# Load validation dataset
#
import xarray as xr

fname = checkpoint['FNAME_VAL']
varkey = checkpoint['VARKEY']

## Loading raw data and normaliation
ds = xr.open_dataset(fname)
da = ds[varkey]

dataset = EnsembleDataset(da, transform)
loader  = DataLoader(dataset, batch_size=12,shuffle=False)

In [ ]:
#
# Reconstruction of normalized variable
#
model.eval()
reconstructions = []
targets = []

with torch.no_grad():
    for batch in loader:
        batch = batch.to(device)
        
        recon = model(batch)
        
        reconstructions.append(recon.detach().cpu())
        targets.append(batch.detach().cpu())

reconstructions = torch.cat(reconstructions, dim=0).squeeze(1)
targets = torch.cat(targets, dim=0).squeeze(1)

In [ ]:
#
# Save reconstructions and targets
#
nens = reconstructions.shape[0]
data_vars = {
    "reconstruction": (("ens", "lat", "lon"), reconstructions.numpy()),
    "target": (("ens", "lat", "lon"), targets.numpy()),
}
ds = xr.Dataset(
    data_vars,
    coords={"ens": range(nens)},
)
ds.to_netcdf(output_dir / "reconstruction.nc")